## 1. Import & cấu hình
Cập nhật: bỏ `%pylab inline` (deprecated), dùng `plt.rcParams` trực tiếp.

**Fix**: logic ingest (download GCS, đọc CSV, parse basic features) đã được
tách ra `src/ingest.py` để tái sử dụng được — notebook này giờ chỉ import và
gọi, không còn tự viết logic đọc dữ liệu inline nữa.

In [ ]:
import sys
sys.path.insert(0, "..")  # để import được src/ khi chạy notebook từ notebooks/

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.ingest import download_from_gcs, peek_schema, load_raw, parse_basic_features

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline

## 2. Đọc dữ liệu

### 2.1 Đọc thử 10 dòng đầu (train & test)
Peek nhanh trước khi đọc full/sample, để kiểm tra schema và cột khác nhau giữa train/test.

In [ ]:
BUCKET_NAME = "TEN-BUCKET-CUA-BAN"  # TODO: điền tên bucket thật
PROJECT = "santander-ds"
TRAIN_BLOB = "raw/train_ver2.csv"
TEST_BLOB = "raw/test_ver2.csv"

train_peek = peek_schema(BUCKET_NAME, TRAIN_BLOB, project=PROJECT, nrows=10)
test_peek = peek_schema(BUCKET_NAME, TEST_BLOB, project=PROJECT, nrows=10)

print("Train shape (10 dòng):", train_peek.shape)
print("Test shape (10 dòng):", test_peek.shape)

print("\nCột chỉ có ở train:", set(train_peek.columns) - set(test_peek.columns))
print("Cột chỉ có ở test:", set(test_peek.columns) - set(train_peek.columns))

In [ ]:
train_peek.head(10)

In [ ]:
test_peek.head(10)

### 2.2 Đọc dữ liệu train (sample để tránh crash kernel)
Tải từ GCS bucket rồi đọc, giới hạn số dòng và sample khách hàng để tránh crash kernel.
Đổi `BUCKET_NAME`/`LIMIT_ROWS`/`LIMIT_PEOPLE` cho phù hợp.

In [ ]:
LIMIT_ROWS   = 10_000_000
LIMIT_PEOPLE = 10_000
RANDOM_STATE = 42

train_csv_path = download_from_gcs(
    BUCKET_NAME, TRAIN_BLOB, "data/raw/train_ver2.csv", project=PROJECT
)
df = load_raw(train_csv_path, limit_rows=LIMIT_ROWS, limit_people=LIMIT_PEOPLE,
              random_state=RANDOM_STATE)

df.describe()

### 2.3 Parse ngày tháng & feature cơ bản
`fecha_dato` là ngày của dòng dữ liệu, `fecha_alta` là ngày khách hàng gia nhập.
Thêm cột `month` vì hành vi mua sản phẩm có thể phụ thuộc thời điểm trong năm.

In [ ]:
df = parse_basic_features(df)

df["fecha_dato"].unique()

## 3. EDA — dữ liệu thô
Trước khi ra quyết định impute/xử lý outlier cho từng cột, xem tổng quan toàn bộ dataset để quyết định có căn cứ, không phải "thấy lỗi thì sửa".

### 3.1 Tổng quan dataset

In [ ]:
print(f"Dataset shape: {df.shape}")
print(f"Số khách hàng: {df['ncodpers'].nunique()}")
print(f"Khoảng thời gian: {df['fecha_dato'].min()} -> {df['fecha_dato'].max()}")
df.dtypes.value_counts()

### 3.2 Tỷ lệ missing toàn bộ

In [ ]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

plt.figure(figsize=(10, max(4, len(missing_pct) * 0.35)))
sns.barplot(x=missing_pct.values, y=missing_pct.index, color="steelblue")
plt.xlabel("% missing")
plt.title("Tỷ lệ thiếu dữ liệu theo cột (toàn bộ dataset)")
plt.tight_layout()
plt.show()

missing_pct

### 3.3 Số dòng trùng

In [ ]:
n_dup_full = df.duplicated().sum()
n_dup_key = df.duplicated(subset=["ncodpers", "fecha_dato"]).sum()
print(f"Số dòng trùng hoàn toàn: {n_dup_full}")
print(f"Số dòng trùng theo (ncodpers, fecha_dato): {n_dup_key}")

### 3.4 Phân phối các biến numeric chính

In [ ]:
key_numeric = [c for c in ["age", "antiguedad", "renta", "indrel", "ind_actividad_cliente"] if c in df.columns]

for col in key_numeric:
    plt.figure(figsize=(10, 6))
    sns.histplot(pd.to_numeric(df[col], errors="coerce").dropna(), bins=50, color="slateblue")
    plt.title(col, fontsize=16)
    plt.xlabel(col, fontsize=13)
    plt.ylabel("Count", fontsize=13)
    plt.tight_layout()
    plt.show()

### 3.5 Tổng quan các biến categorical

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
for col in categorical_cols:
    print(f"{col}: {df[col].nunique(dropna=True)} giá trị duy nhất")

low_card_cols = [c for c in categorical_cols if df[c].nunique(dropna=True) <= 10]
for col in low_card_cols:
    plt.figure(figsize=(5, 3))
    df[col].value_counts(dropna=False).plot(kind="bar", color="coral")
    plt.title(col)
    plt.tight_layout()
    plt.show()

### 3.6 Kiểm tra missing có ngẫu nhiên không (renta theo tỉnh)

In [ ]:
df["renta_missing"] = df["renta"].isnull()
missing_rate_by_province = df.groupby("nomprov")["renta_missing"].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(x=missing_rate_by_province.index, y=missing_rate_by_province.values, color="darkorange")
plt.xticks(rotation=90)
plt.ylabel("Tỷ lệ thiếu renta")
plt.title("Tỷ lệ missing renta theo tỉnh — missing có phải ngẫu nhiên không?")
plt.tight_layout()
plt.show()

df.drop(columns=["renta_missing"], inplace=True)

### 3.7 Tương quan giữa các biến numeric chính

In [ ]:
corr = df[key_numeric].apply(pd.to_numeric, errors="coerce").corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Tương quan giữa các biến numeric chính")
plt.tight_layout()
plt.show()

### 3.8 Số dòng dữ liệu theo tháng

In [ ]:
records_per_month = df["fecha_dato"].value_counts().sort_index()
plt.figure(figsize=(10, 4))
records_per_month.plot(kind="bar", color="teal")
plt.title("Số dòng dữ liệu theo tháng")
plt.ylabel("Số khách hàng")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.9 Phân phối số sản phẩm sở hữu / khách hàng / tháng
(Gần nhất với "target" của bài toán.)

In [ ]:
product_cols = [c for c in df.columns if c.startswith("ind_") and c.endswith("_ult1")]
df["n_products"] = df[product_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

plt.figure(figsize=(8, 4))
sns.histplot(df["n_products"], bins=range(0, int(df["n_products"].max()) + 2), color="mediumseagreen")
plt.title("Phân phối số sản phẩm sở hữu / khách hàng / tháng")
plt.xlabel("Số sản phẩm")
plt.tight_layout()
plt.show()

df.drop(columns=["n_products"], inplace=True)

## Checkpoint — lưu cho notebook tiếp theo
Lưu lại `df` (đã đọc + parse ngày/basic feature, CHƯA làm sạch) để
`02_cleaning.ipynb` đọc tiếp — không cần ingest lại từ GCS mỗi lần chạy notebook khác.

In [ ]:
import os
os.makedirs("data/interim", exist_ok=True)
df.to_parquet("data/interim/raw_parsed.parquet", index=False)
print(f"Đã lưu checkpoint: {df.shape}")